In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index


documents = load_faq_data()
index = build_index(documents) 

assistant = RAGBase(index=index, llm_client=openai_client)

In [3]:
print(assistant.rag("How do I run Ollama locally?"))

To run Ollama locally, install it first from [https://ollama.com/download](https://ollama.com/download):

- **macOS**: download and install the `.pkg`
- **Windows**: download and install the `.msi`
- **Linux**: run
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Then start a local model with:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat interface.

To verify the local server is running, you can test:

```bash
curl http://localhost:11434
```

If you want to use it from Python, install the client:

```bash
pip install ollama
```

and then call it like:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [4]:
print(assistant.rag("How do I run Olama locally?"))

I don't know.


In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search("How do I run Ollama locally?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npi

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
messages = [
    {'role': "user", 'content': "I just discoverd the course, Can I join it?"}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [15]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment discover course join"}', call_id='call_8kTTmwXK5zq5SvcpzcKei9UN', name='search', type='function_call', id='fc_0942dbeeb7e8f9c3006a2e22bad24c81998593fe28b3c299c9', namespace=None, status='completed')

In [16]:
response.output[0].arguments

'{"query":"Can I join the course after it has started? late enrollment discover course join"}'

In [17]:
call = response.output[0]

In [21]:
import json

args = json.loads(call.arguments)

In [22]:
call.name

'search'

In [25]:
results = search(**args)
result_json = json.dumps(results, indent=2)

In [32]:
print(results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zo

In [33]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [27]:
messages.extend(response.output)

In [28]:
messages

[{'role': 'user', 'content': 'I just discoverd the course, Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment discover course join"}', call_id='call_8kTTmwXK5zq5SvcpzcKei9UN', name='search', type='function_call', id='fc_0942dbeeb7e8f9c3006a2e22bad24c81998593fe28b3c299c9', namespace=None, status='completed')]

In [29]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [30]:
messages

[{'role': 'user', 'content': 'I just discoverd the course, Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment discover course join"}', call_id='call_8kTTmwXK5zq5SvcpzcKei9UN', name='search', type='function_call', id='fc_0942dbeeb7e8f9c3006a2e22bad24c81998593fe28b3c299c9', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_8kTTmwXK5zq5SvcpzcKei9UN',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode an

In [31]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [34]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(657, 31)

In [35]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
